# Random Forest — Walmart Store Sales Forecasting (v2 FE)

## Goal
Forecast weekly `Weekly_Sales` with **RandomForestRegressor**. Metric = **WMAE** (holiday weight **5**).

## Feature engineering (v2)
Rich FE inspired by teammate notes, implemented **leakage-safe**:
- IQR clip on numeric features (bounds from **train only**)
- calendar: sin/cos week, `Weeks_elapsed`, `WeekOfMonth`
- holiday: signed distance + per-holiday `Days_to_*` / `Days_post_*`
- lags / rolling / YoY ratio from **past** sales only (`shift`)
- expanding target encodings (`shift(1)` — no current-row target)
- richer MarkDown stats + holiday×markdown + Unemployment×markdown

## Kaggle
1. Add competition data  
2. **CPU** · **Internet On**  
3. Prefer **Save Version → Save & Run All** (and/or MLflow artifacts)

Output: `submission_randomforest.csv`

In [ ]:
#1 — setup (CPU)
!pip install -q dagshub mlflow pyarrow

import os, warnings, itertools, json
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
import mlflow

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

NOTEBOOK_VERSION = 'randomforest_v2_fe'
RANDOM_STATE = 42
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

USE_MLFLOW = False
try:
    import dagshub
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('DAGSHUB_USER_TOKEN')
    except Exception:
        token = os.environ.get('DAGSHUB_USER_TOKEN')
    if token:
        os.environ['DAGSHUB_USER_TOKEN'] = token
        dagshub.auth.add_app_token(token)
        dagshub.init(
            repo_owner='lshek22',
            repo_name='walmart-recruiting-store-sales-forecasting',
            mlflow=True,
        )
        USE_MLFLOW = True
        print('DagsHub/MLflow: token auth OK')
    else:
        print('DagsHub token missing — MLflow disabled (optional)')
except Exception as e:
    print(f'MLflow disabled ({type(e).__name__}: {e})')

if USE_MLFLOW:
    EXP_CLEANING = 'RandomForest_Cleaning'
    EXP_FEATURES = 'RandomForest_Feature_Selection'
    EXP_TRAINING = 'RandomForest_Training'
else:
    EXP_CLEANING = EXP_FEATURES = EXP_TRAINING = 'RandomForest_Local'

    @contextmanager
    def _noop_run(*args, **kwargs):
        class _Dummy:
            def __enter__(self): return self
            def __exit__(self, *a): return False
            def __getattr__(self, name):
                return lambda *a, **k: None
        yield _Dummy()

    mlflow.start_run = _noop_run
    mlflow.set_experiment = lambda *a, **k: None
    mlflow.log_params = lambda *a, **k: None
    mlflow.log_param = lambda *a, **k: None
    mlflow.log_metric = lambda *a, **k: None
    mlflow.log_artifact = lambda *a, **k: None

print(f'Ready | {NOTEBOOK_VERSION} | mlflow={USE_MLFLOW}')

## 1 — Load, clean, light EDA

Clip negative sales (returns). Feature outlier clipping happens later using **train-only** IQR bounds.

In [ ]:
#2 — load + clean
COMPETITION_SLUG = 'walmart-recruiting-store-sales-forecasting'
MARKDOWN_COLS = [f'MarkDown{i}' for i in range(1, 6)]


def find_competition_data_dir():
    preferred = f'/kaggle/input/competitions/{COMPETITION_SLUG}'
    if os.path.isdir(preferred):
        return preferred
    for root, _, files in os.walk('/kaggle/input'):
        if {'train.csv', 'train.csv.zip'} & set(files):
            return root
    for cand in (Path('.'), Path('..'), Path('../data')):
        if (cand / 'train.csv').exists() or (cand / 'train.csv.zip').exists():
            return str(cand.resolve())
    raise FileNotFoundError('Competition data not found')


def read_competition_csv(data_dir, stem, parse_dates=None):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path, parse_dates=parse_dates)
    raise FileNotFoundError(f'Missing {stem} in {data_dir}')


def load_raw(data_dir):
    train = read_competition_csv(data_dir, 'train', parse_dates=['Date'])
    test = read_competition_csv(data_dir, 'test', parse_dates=['Date'])
    stores = read_competition_csv(data_dir, 'stores')
    features = read_competition_csv(data_dir, 'features', parse_dates=['Date'])
    return train, test, stores, features


def clean_and_merge(train, test, stores, features):
    feat = features.drop(columns=['IsHoliday'])

    def merge_one(df):
        out = df.merge(stores, on='Store', how='left')
        out = out.merge(feat, on=['Store', 'Date'], how='left')
        out[MARKDOWN_COLS] = out[MARKDOWN_COLS].fillna(0)
        out['CPI'] = out.groupby('Store')['CPI'].transform(lambda s: s.ffill().bfill())
        out['Unemployment'] = out.groupby('Store')['Unemployment'].transform(lambda s: s.ffill().bfill())
        return out

    return merge_one(train), merge_one(test)


DATA_DIR = find_competition_data_dir()
print('DATA_DIR:', DATA_DIR)

mlflow.set_experiment(EXP_CLEANING)
with mlflow.start_run(run_name='clean_merge_rf_v2'):
    train_raw, test_raw, stores_raw, features_raw = load_raw(DATA_DIR)
    n_neg = int((train_raw['Weekly_Sales'] < 0).sum())
    train_df, test_df = clean_and_merge(train_raw, test_raw, stores_raw, features_raw)
    train_df['Weekly_Sales'] = train_df['Weekly_Sales'].clip(lower=0)

    mlflow.log_params({
        'notebook_version': NOTEBOOK_VERSION,
        'negative_sales_clip': True,
        'fe_source': 'friend_advice_leakage_safe_v2',
    })
    mlflow.log_metric('n_train', len(train_df))
    mlflow.log_metric('n_test', len(test_df))
    mlflow.log_metric('n_negative_sales_raw', n_neg)

print(train_df.shape, test_df.shape, '| neg sales clipped from', n_neg)

# light EDA
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='#8b949e')
    ax.title.set_color('white')

axes[0].hist(train_df['Weekly_Sales'].clip(upper=train_df['Weekly_Sales'].quantile(0.99)), bins=40, color='#58a6ff')
axes[0].set_title('Weekly_Sales (clip 99pct for view)')
hol = train_df.groupby('IsHoliday')['Weekly_Sales'].mean()
axes[1].bar(['Non-holiday', 'Holiday'], hol.values, color=['#8b949e', '#f78166'])
axes[1].set_title('Mean sales by IsHoliday')
axes[2].plot(train_df.groupby('Date')['Weekly_Sales'].mean(), color='#3fb950', lw=1)
axes[2].set_title('Mean sales over time')
plt.tight_layout()
plt.savefig('eda_rf_v2.png', dpi=120, facecolor='#0f1117')
plt.show()
train_df.head()

## 2 — Feature engineering (v2, leakage-safe)

| Idea from friend | How we implement |
|---|---|
| IQR outlier clip | Bounds from **train**; apply to train+test |
| Week sin/cos | Keep |
| Weeks_elapsed / WeekOfMonth | Add |
| Signed holiday + per-holiday distances | `Holiday_signed`, `Days_to_*`, `Days_post_*` |
| Lags / rolling / YoY | `shift` / `shift+rolling` only (no current y) |
| Target encoding | Expanding mean with `shift(1)` |
| store_dept_week mean | Expanding mean within Store–Dept–WOY + `shift(1)` |
| Richer MarkDown + interactions | `MD_max`, `MD_nonzero_mean`, holiday×MD, Unemp×MD |

Lags need train+test stacked in time so test rows can see **past train sales**.

In [ ]:
#3 — feature engineering v2
HOLIDAYS = {
    'SuperBowl': pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']),
    'LaborDay': pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']),
    'Thanksgiving': pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']),
    'Christmas': pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']),
}
ALL_HOLIDAY_DATES = pd.to_datetime(np.concatenate([d.values for d in HOLIDAYS.values()]))

CLIP_COLS = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'] + MARKDOWN_COLS


def fit_iqr_bounds(train_df, cols, k=1.5):
    # friend ~±4.5σ ≈ aggressive; IQR with larger k keeps spirit without nuking tails
    bounds = {}
    for c in cols:
        s = train_df[c].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        # k=4.5 on IQR ≈ very wide fences (closer to friend's 4.5σ idea)
        lo, hi = q1 - 4.5 * iqr, q3 + 4.5 * iqr
        bounds[c] = (float(lo), float(hi))
    return bounds


def apply_iqr_bounds(df, bounds):
    out = df.copy()
    for c, (lo, hi) in bounds.items():
        if c in out.columns:
            out[c] = out[c].clip(lo, hi)
    return out


def _days_to_nearest(dates, holiday_dates):
    # signed: negative = before holiday, positive = after (friend isSigned idea)
    d = pd.to_datetime(dates).values.astype('datetime64[D]')
    h = holiday_dates.values.astype('datetime64[D]')
    signed = np.empty(len(d), dtype=np.int32)
    for i in range(0, len(d), 5000):
        sl = d[i:i + 5000]
        # delta = date - holiday: negative if date is before holiday
        delta = (sl[:, None] - h[None, :]).astype('timedelta64[D]').astype(int)
        j = np.argmin(np.abs(delta), axis=1)
        signed[i:i + 5000] = delta[np.arange(len(sl)), j]
    return signed


def add_calendar_holiday_markdown(df):
    out = df.copy()
    out['Year'] = out['Date'].dt.year
    out['Month'] = out['Date'].dt.month
    out['WeekOfYear'] = out['Date'].dt.isocalendar().week.astype(int)
    out['DayOfYear'] = out['Date'].dt.dayofyear
    out['Week_sin'] = np.sin(2 * np.pi * out['WeekOfYear'] / 52)
    out['Week_cos'] = np.cos(2 * np.pi * out['WeekOfYear'] / 52)
    out['WeekOfMonth'] = ((out['Date'].dt.day - 1) // 7 + 1).astype(int)
    out['IsHoliday'] = out['IsHoliday'].astype(int)
    out['Type_enc'] = out['Type'].map({'A': 0, 'B': 1, 'C': 2})

    # MarkDown aggregates
    md = out[MARKDOWN_COLS]
    out['MD_total'] = md.sum(axis=1)
    out['MD_count'] = (md > 0).sum(axis=1)
    out['MD_max'] = md.max(axis=1)
    nonzero = md.where(md > 0)
    out['MD_nonzero_mean'] = nonzero.mean(axis=1).fillna(0)
    out['MD_holiday'] = out['MD_total'] * out['IsHoliday']
    out['Unemp_x_MD_count'] = out['Unemployment'] * out['MD_count']

    # holiday distances
    out['Holiday_signed'] = _days_to_nearest(out['Date'], ALL_HOLIDAY_DATES)
    out['Days_to_holiday'] = np.abs(out['Holiday_signed'])
    for name, hdates in HOLIDAYS.items():
        signed = _days_to_nearest(out['Date'], hdates)
        # before holiday: signed < 0 → Days_to = -signed; after → Days_post = signed
        out[f'Days_to_{name}'] = np.clip(-signed, 0, None)
        out[f'Days_post_{name}'] = np.clip(signed, 0, None)
        out[f'Signed_{name}'] = signed

    return out


def add_lag_roll_te(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()
    te['Weekly_Sales'] = np.nan  # never use test target (none anyway)

    both = pd.concat([tr, te], ignore_index=True)
    both = both.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

    origin = both['Date'].min()
    both['Weeks_elapsed'] = ((both['Date'] - origin).dt.days // 7).astype(int)

    g = both.groupby(['Store', 'Dept'], sort=False)

    for lag in [1, 2, 4, 8, 13, 26, 52]:
        both[f'lag_{lag}'] = g['Weekly_Sales'].shift(lag)

    # rolling on past only
    both['roll_mean_4'] = g['Weekly_Sales'].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    both['roll_mean_8'] = g['Weekly_Sales'].transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())
    both['roll_std_4'] = g['Weekly_Sales'].transform(lambda s: s.shift(1).rolling(4, min_periods=2).std())
    both['roll_mean_4_ly'] = g['Weekly_Sales'].transform(lambda s: s.shift(53).rolling(4, min_periods=1).mean())
    both['yoy_roll4_ratio'] = both['roll_mean_4'] / both['roll_mean_4_ly'].replace(0, np.nan)

    # expanding target encodings (past only)
    both = both.sort_values(['Date', 'Store', 'Dept']).reset_index(drop=True)
    both['te_store'] = both.groupby('Store')['Weekly_Sales'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    both['te_dept'] = both.groupby('Dept')['Weekly_Sales'].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
    both['te_store_dept'] = both.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    # historical same week-of-year for store-dept (previous years)
    both['te_store_dept_woy'] = both.groupby(['Store', 'Dept', 'WeekOfYear'])['Weekly_Sales'].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )

    both = both.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    train_out = both[both['Weekly_Sales'].notna()].copy()
    test_out = both[both['Weekly_Sales'].isna()].copy()
    # restore: test rows were marked nan sales — drop target col for clarity
    if 'Weekly_Sales' in test_out.columns:
        test_out = test_out.drop(columns=['Weekly_Sales'])
    return train_out, test_out


print('Fitting IQR bounds on train...')
iqr_bounds = fit_iqr_bounds(train_df, CLIP_COLS)
train_c = apply_iqr_bounds(train_df, iqr_bounds)
test_c = apply_iqr_bounds(test_df, iqr_bounds)

print('Calendar / holiday / markdown features...')
train_c = add_calendar_holiday_markdown(train_c)
test_c = add_calendar_holiday_markdown(test_c)

print('Lags / rolling / target encodings (train+test stacked)...')
train_fe, test_fe = add_lag_roll_te(train_c, test_c)
print('train_fe', train_fe.shape, '| test_fe', test_fe.shape)

candidate_features = [
    # ids / store
    'Store', 'Dept', 'Type_enc', 'Size',
    # macro
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    # markdown
    'MD_total', 'MD_count', 'MD_max', 'MD_nonzero_mean', 'MD_holiday', 'Unemp_x_MD_count',
    # calendar
    'Year', 'Month', 'WeekOfYear', 'Week_sin', 'Week_cos', 'WeekOfMonth', 'Weeks_elapsed',
    # holiday
    'IsHoliday', 'Holiday_signed', 'Days_to_holiday',
    'Days_to_SuperBowl', 'Days_post_SuperBowl', 'Signed_SuperBowl',
    'Days_to_LaborDay', 'Days_post_LaborDay', 'Signed_LaborDay',
    'Days_to_Thanksgiving', 'Days_post_Thanksgiving', 'Signed_Thanksgiving',
    'Days_to_Christmas', 'Days_post_Christmas', 'Signed_Christmas',
    # lags / roll
    'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_13', 'lag_26', 'lag_52',
    'roll_mean_4', 'roll_mean_8', 'roll_std_4', 'roll_mean_4_ly', 'yoy_roll4_ratio',
    # encodings
    'te_store', 'te_dept', 'te_store_dept', 'te_store_dept_woy',
]

missing = [c for c in candidate_features if c not in train_fe.columns]
assert not missing, missing
print('n_candidate_features', len(candidate_features))
train_fe[candidate_features[:12] + ['Weekly_Sales']].head()

In [ ]:
#4 — fill NaNs (sklearn RF cannot take NaN) + probe importance
IMPORTANCE_THRESHOLD = 0.002  # slightly lower: more features now

fill_values = train_fe[candidate_features].median(numeric_only=True)

def fill_features(df):
    out = df.copy()
    for c in candidate_features:
        out[c] = out[c].fillna(fill_values.get(c, 0))
    return out

train_fe = fill_features(train_fe)
test_fe = fill_features(test_fe)

mlflow.set_experiment(EXP_FEATURES)
with mlflow.start_run(run_name='feature_engineering_rf_v2'):
    mlflow.log_param('n_candidate_features', len(candidate_features))
    mlflow.log_param('notebook_version', NOTEBOOK_VERSION)

    X_probe = train_fe[candidate_features]
    y_probe = train_fe['Weekly_Sales']
    w_probe = np.where(train_fe['IsHoliday'] == 1, 5, 1)

    probe_model = RandomForestRegressor(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    print('Fitting probe RF for importance...', flush=True)
    probe_model.fit(X_probe, y_probe, sample_weight=w_probe)

    importances = pd.Series(probe_model.feature_importances_, index=candidate_features).sort_values(ascending=False)
    selected_features = importances[importances >= IMPORTANCE_THRESHOLD].index.tolist()
    dropped_features = importances[importances < IMPORTANCE_THRESHOLD].index.tolist()
    if len(selected_features) < 10:
        selected_features = importances.head(30).index.tolist()
        dropped_features = [c for c in candidate_features if c not in selected_features]

    mlflow.log_param('n_selected_features', len(selected_features))
    mlflow.log_param('selected_features', ', '.join(selected_features))

    fig, ax = plt.subplots(figsize=(8, max(6, len(importances) * 0.18)))
    importances.plot(kind='barh', ax=ax, color='#58a6ff')
    ax.invert_yaxis()
    ax.set_title('RF probe importance (v2 FE)')
    fig.tight_layout()
    fig.savefig('feature_importance_probe_rf_v2.png', dpi=130)
    mlflow.log_artifact('feature_importance_probe_rf_v2.png')
    plt.show()

print('Selected', len(selected_features), 'features')
print('Dropped', dropped_features)
importances.head(20)

## 3 — Time-series CV + Random Forest

Same week-level `TimeSeriesSplit` as before. Slightly smaller grid because FE is heavier.

In [ ]:
#5 — CV grid search

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(np.asarray(is_holiday) == 1, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


N_SPLITS = 4
# Keep previous v1 winner in the grid (depth=24, max_features=0.8), plus a few neighbors for new FE
param_grid = {
    'n_estimators': [300],
    'max_depth': [24, 28],
    'min_samples_leaf': [2, 5],
    'max_features': [0.8, 'sqrt'],
}
grid_keys = list(param_grid.keys())
grid_combos = list(itertools.product(*param_grid.values()))
print(f'Grid: {len(grid_combos)} configs × {N_SPLITS} folds (includes v1 winner 24/0.8/leaf2)')

X_all = train_fe[selected_features]
y_all = train_fe['Weekly_Sales']
holiday_all = train_fe['IsHoliday']
dates_all = train_fe['Date']
unique_dates = np.sort(dates_all.unique())
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

mlflow.set_experiment(EXP_TRAINING)
results = []

with mlflow.start_run(run_name='rf_time_cv_search_v2'):
    mlflow.log_param('n_splits', N_SPLITS)
    mlflow.log_param('n_features', len(selected_features))
    mlflow.log_param('notebook_version', NOTEBOOK_VERSION)

    for combo_i, combo in enumerate(grid_combos, 1):
        params = dict(zip(grid_keys, combo))
        fold_scores = []
        print(f'\n[{combo_i}/{len(grid_combos)}] {params}', flush=True)

        with mlflow.start_run(run_name=f'grid_{params}', nested=True):
            mlflow.log_params({k: str(v) for k, v in params.items()})
            for fold, (tr_i, va_i) in enumerate(tscv.split(unique_dates)):
                tr_dates, va_dates = unique_dates[tr_i], unique_dates[va_i]
                tr_m = dates_all.isin(tr_dates)
                va_m = dates_all.isin(va_dates)
                model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)
                model.fit(
                    X_all[tr_m], y_all[tr_m],
                    sample_weight=np.where(holiday_all[tr_m] == 1, 5, 1),
                )
                preds = model.predict(X_all[va_m])
                score = wmae(y_all[va_m].values, preds, holiday_all[va_m].values)
                fold_scores.append(score)
                mlflow.log_metric('fold_wmae', score, step=fold)
                print(f'  fold {fold}: {score:,.2f}', flush=True)

            mean_wmae, std_wmae = float(np.mean(fold_scores)), float(np.std(fold_scores))
            mlflow.log_metric('mean_cv_wmae', mean_wmae)
            mlflow.log_metric('std_cv_wmae', std_wmae)

        results.append({**params, 'mean_cv_wmae': mean_wmae, 'std_cv_wmae': std_wmae})
        print(f'  -> mean {mean_wmae:,.2f} +/- {std_wmae:,.2f}', flush=True)

    results_df = pd.DataFrame(results).sort_values('mean_cv_wmae').reset_index(drop=True)
    best_row = results_df.iloc[0]
    best_params = {
        'n_estimators': int(best_row['n_estimators']),
        'max_depth': int(best_row['max_depth']),
        'min_samples_leaf': int(best_row['min_samples_leaf']),
        'max_features': best_row['max_features'],
    }
    mlflow.log_param('best_params', str(best_params))
    mlflow.log_metric('best_mean_cv_wmae', float(best_row['mean_cv_wmae']))
    results_df.to_csv('cv_grid_results_rf_v2.csv', index=False)
    mlflow.log_artifact('cv_grid_results_rf_v2.csv')

    print('\nFitting final model on all train...', flush=True)
    final_model = RandomForestRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
    final_model.fit(X_all, y_all, sample_weight=np.where(holiday_all == 1, 5, 1))
    train_wmae = wmae(y_all.values, np.clip(final_model.predict(X_all), 0, None), holiday_all.values)
    mlflow.log_metric('final_train_wmae', train_wmae)

    fig, ax = plt.subplots(figsize=(8, max(5, len(selected_features) * 0.2)))
    pd.Series(final_model.feature_importances_, index=selected_features).sort_values().plot(kind='barh', ax=ax, color='#f78166')
    ax.set_title('Final RF importance (v2 FE)')
    fig.tight_layout()
    fig.savefig('final_feature_importance_rf_v2.png', dpi=130)
    mlflow.log_artifact('final_feature_importance_rf_v2.png')
    plt.show()

    with open('rf_best_pipeline_spec_v2.json', 'w', encoding='utf-8') as f:
        json.dump({
            'notebook_version': NOTEBOOK_VERSION,
            'best_params': best_params,
            'selected_features': selected_features,
            'best_mean_cv_wmae': float(best_row['mean_cv_wmae']),
            'final_train_wmae': float(train_wmae),
        }, f, indent=2)
    mlflow.log_artifact('rf_best_pipeline_spec_v2.json')

print('Best:', best_params, '| CV WMAE', round(best_row['mean_cv_wmae'], 2))
results_df

## 4 — Submission

Save CSV **and** log to MLflow (so you can download from DagsHub if `/kaggle/working` is empty later).

In [ ]:
#6 — submission
X_test = test_fe[selected_features]
test_preds = pd.Series(np.clip(final_model.predict(X_test), 0, None))
test_preds = test_preds.replace([np.inf, -np.inf], np.nan).fillna(float(y_all.median()))

submission = pd.DataFrame({
    'Id': (
        test_fe['Store'].astype(str) + '_' +
        test_fe['Dept'].astype(str) + '_' +
        test_fe['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': test_preds.to_numpy(),
})
assert submission['Weekly_Sales'].notna().all()
assert len(submission) == len(test_fe)

sub_path = WORK_DIR / 'submission_randomforest.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved {sub_path} | rows={len(submission)}')
print(submission.head())

with mlflow.start_run(run_name='RandomForest_Submission_v2'):
    mlflow.log_params({k: str(v) for k, v in best_params.items()})
    mlflow.log_param('notebook_version', NOTEBOOK_VERSION)
    mlflow.log_param('n_features', len(selected_features))
    # proper model quality metrics (not just row count)
    mlflow.log_metric('best_mean_cv_wmae', float(best_row['mean_cv_wmae']))
    mlflow.log_metric('best_std_cv_wmae', float(best_row['std_cv_wmae']))
    mlflow.log_metric('final_train_wmae', float(train_wmae))
    mlflow.log_metric('n_submission_rows', len(submission))
    mlflow.log_artifact(str(sub_path))

try:
    from IPython.display import FileLink, display
    display(FileLink(sub_path.name))
except Exception:
    pass

print('Also download from MLflow artifacts if working dir is empty later.')
print('Upload submission_randomforest.csv → Kaggle')